In [1]:
print(1243)

1243


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32332397)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.019730438)

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
len(texts)

1380

In [11]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1380

In [13]:
import numpy as np
X = np.array(vectors)

X.shape

(1380, 384)

In [14]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [16]:
scores = X.dot(v_query)

scores

array([ 0.23396154,  0.05057262,  0.00561457, ...,  0.0954106 ,
       -0.04206598,  0.03490689], shape=(1380,), dtype=float32)

In [18]:
idx = np.argmax(scores)
idx, scores[idx], documents[idx]["question"], documents[idx]["answer"]

(np.int64(860),
 np.float32(0.762941),
 'Course: Can I still join the course after the start date?',
 "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.")

In [20]:
# Top 5
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1] # reverse to get descending order
top5

array([ 860,  473,   29, 1262,  865])

In [21]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [ ]:
top5 = np.argsort(-scores)[:5] # -scores = negative numbers, largest -> smallest

In [23]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [27]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [28]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [34]:
!wget https://raw.githubusercontent.com/PhilN8/llm-zoomcamp-2026-module-1/main/rag_helper.py

--2026-07-23 11:30:22--  https://raw.githubusercontent.com/PhilN8/llm-zoomcamp-2026-module-1/main/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2158 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.11K  --.-KB/s    in 0s      

2026-07-23 11:30:22 (30.4 MB/s) - ‘rag_helper.py’ saved [2158/2158]



In [30]:
!uv add google-genai

Resolved 149 packages in 5.39s                                       
⠙ typing-extensions==4.16.0                                                     Checked 144 packages in 756ms


In [31]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [36]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [41]:
from rag_helper import RAGBase, RAGVector

assistant = RAGBase(
    index=index,
    llm_client=client,
)

ImportError: cannot import name 'RAGVector' from 'rag_helper' (/workspaces/llm-zoomcamp-2026-module-2/rag_helper.py)

In [39]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

[ThoughtStep(signature='EuwICukIARFNMg802oab8S+zpStW6NlhNNpS1iENooHgGW3lAFZg6HBS0v/+Dw7ww69xhyEKIYmG+C/ALexDdRyBSlsozmRAKQ+qPo4AMOn72gL5KEr8BnZ14HKqDQ/ef6cwTPvNk+lF9BHc/qIrQFi/A4W5bJ27+dcijtJz9mbcPUM4pvCrh7RSetw7m8QqizMEhEn7xpXIaB2h5BzC4NNITcxdFgvenGpxVLrJPHddHcO8z+807NkTOxnhs07hfSMIsPAqaM2s/ISMT2+0pXjfEdbsjcyBg92fqStAGKjoVYCugr34w66XmnWAZ8o/1LcriNcbQajbajx+swjmyI+PKqnHV6mrEarFWUgImgGT7vFRs/Yr6tyS5+gk5goQ9DQJ2v/B8UIdlUaE9O1vn2osRk/7/26eg2d3eyNdnPwC6hcjtxC3yFndBs2+pJn1IKC/TnXtm3EX9scehyww4k5kLXMACIATKUrlyjiuAY0gPi9jNZU+U6lU2Ee5DYKa9vakYgyIYAnYpojvKQcUWkWX9pBSuFEZhhjagjqbSv/Um053xeBTHD3J50+jFtZYvQ/4BgHvXl/KhnSC0DgTcuZMc5InfoPpj6aBLolVMGSiF2rve2jL1nik4/puxQPNaWbYvJMK9YfQhi32H8LQliFQlN7LJVlGD/VUngNJSJTaSYFhZCEwM9MD+p90urQChFikQzr2wabAh3qgaE92wndnWjVD2PV5NMawS73Y7AIbyBff8rJ5FpEQOKQzrisayWSkymZ4JFvr0NwxWqIxAYzPATuVbpcbKf1Lk2szRbEmOb2PZ6s/kDSBnf/j4sWHk93YfOmqPLYPgoyDH/3qrPceI+JCzNpmFISbEzNKIWcKFldJN9UI6HRyc6uLhn4unSYYKGiQ+J5q+hfsGt84xK5QOX+j2F7GPhuFgddQFxwHYjxpzGeU6W7J/daIqB6ImJjxvnFuXYba4fwC

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)